In [6]:
import pandas as pd
import geopandas as gpd
from pathlib import Path

def encontrar_raiz(marker="data"):
    p = Path.cwd()
    for _ in range(6):
        if (p / marker).exists():
            return p
        p = p.parent
    raise FileNotFoundError(f"No encontré una carpeta '{marker}' subiendo desde {Path.cwd()}")

RAIZ = encontrar_raiz()
STAGING_DIR = RAIZ / "data" / "clean" / "staging"
assert STAGING_DIR.exists(), f"No existe: {STAGING_DIR}"

archivos = sorted(STAGING_DIR.glob("*.csv")) + sorted(STAGING_DIR.glob("*.gpkg"))
assert len(archivos) > 0, "No se encontró ningún archivo en staging"

dataframes = {}
for f in archivos:
    df = gpd.read_file(f) if f.suffix == ".gpkg" else pd.read_csv(f)
    dataframes[f.stem] = df

for nombre, df in dataframes.items():
    print(f"{nombre:45s} shape={df.shape}")

esa_worldcover_distrital_2021                 shape=(1889, 5)
jrc_distrital                                 shape=(1806, 3)
open_buildings_distrital                      shape=(1889, 6)
renamu_distrital_2021_2024                    shape=(7526, 5)
siaf_gasto_devengado_distrital_2021_2025      shape=(9452, 4)
sien_reunis_distrital_2020_2026               shape=(13029, 9)
srtm_distrital                                shape=(1889, 3)
geobase_distrital                             shape=(1889, 14)


# esa_worldcover (estática, cobertura de suelo):

In [7]:
df = dataframes["esa_worldcover_distrital_2021"]
df["ubigeo"] = df["ubigeo"].astype(str).str.zfill(6)

cols_pct = ["pct_cultivo", "pct_construido", "pct_desnudo", "pct_agua_visible"]
for c in cols_pct:
    print(c, "-> min:", df[c].min(), "max:", df[c].max())

print("Filas:", df.shape[0], "(universo esperado: 1889)")
print("Nulos:", df[cols_pct].isna().sum().sum())
dataframes["esa_worldcover_distrital_2021"] = df

pct_cultivo -> min: 0.0 max: 0.8021325143153261
pct_construido -> min: 1.3720796569145775e-06 max: 0.9996817413118498
pct_desnudo -> min: 0.0 max: 0.9904488733680006
pct_agua_visible -> min: 0.0 max: 0.8937380718940415
Filas: 1889 (universo esperado: 1889)
Nulos: 0


# jrc (agua, tiene MENOS filas que el universo):

In [8]:
df = dataframes["jrc_distrital"]
df["ubigeo"] = df["ubigeo"].astype(str).str.zfill(6)

geobase_ubigeo = dataframes["geobase_distrital"].get("UBIGEO", dataframes["geobase_distrital"].get("ubigeo"))
universo = set(geobase_ubigeo.astype(str).str.zfill(6))
faltantes = universo - set(df["ubigeo"])
print(f"Distritos sin dato JRC: {len(faltantes)}")
print(list(faltantes)[:10], "...")

print(df[["pct_agua_permanente", "pct_agua_estacional"]].describe())
dataframes["jrc_distrital"] = df

Distritos sin dato JRC: 83
['060805', '120412', '010523', '120703', '021504', '090104', '090604', '010119', '120207', '010517'] ...
       pct_agua_permanente  pct_agua_estacional
count          1806.000000          1806.000000
mean              0.213432             0.569356
std               0.218605             0.253046
min               0.000000             0.000000
25%               0.020846             0.372273
50%               0.144211             0.574918
75%               0.345324             0.763620
max               1.000000             1.000000


# open_buildings (edificaciones):

In [9]:
df = dataframes["open_buildings_distrital"]
df["ubigeo"] = df["ubigeo"].astype(str).str.zfill(6)

print("Distritos con area_distrito_km2 <= 0:", (df["area_distrito_km2"] <= 0).sum())
print("Distritos con n_edificios == 0:", (df["n_edificios"] == 0).sum())

chequeo = (df["n_edificios"] / df["area_distrito_km2"] - df["densidad_edificios_km2"]).abs()
print("Máxima diferencia densidad calculada vs reportada:", chequeo.max())

dataframes["open_buildings_distrital"] = df

Distritos con area_distrito_km2 <= 0: 0
Distritos con n_edificios == 0: 0
Máxima diferencia densidad calculada vs reportada: 1.8189894035458565e-12


# renamu (panel municipal, 2021-2024):

In [10]:
df = dataframes["renamu_distrital_2021_2024"]
df["ubigeo"] = df["ubigeo"].astype(str).str.zfill(6)
df.rename(columns={"Año": "anio"}, inplace=True)

conteo_anios = df.groupby("ubigeo")["anio"].nunique()
print("Distritos con menos de 4 años reportados:", (conteo_anios < 4).sum())
print("Años presentes:", sorted(df["anio"].unique()))

print("Valores únicos programa_anemia:", df["programa_anemia"].unique())
print("Valores únicos centro_salud_municipal:", df["centro_salud_municipal"].unique())
print("Nulos en personal_total:", df["personal_total"].isna().sum())

dataframes["renamu_distrital_2021_2024"] = df

Distritos con menos de 4 años reportados: 15
Años presentes: [np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024)]
Valores únicos programa_anemia: [1 0]
Valores únicos centro_salud_municipal: [0 1]
Nulos en personal_total: 190


# siaf (panel de gasto, 2021-2025):

In [11]:
df = dataframes["siaf_gasto_devengado_distrital_2021_2025"]
df["ubigeo"] = df["ubigeo"].astype(str).str.zfill(6)
df.rename(columns={"ANO_EJE": "anio"}, inplace=True)

conteo_anios = df.groupby("ubigeo")["anio"].nunique()
print("Distritos con menos de 5 años reportados:", (conteo_anios < 5).sum())
print("Años presentes:", sorted(df["anio"].unique()))

print("Filas con gasto_total < 0:", (df["gasto_total"] < 0).sum())
print("Filas con gasto_anemia_pan < 0:", (df["gasto_anemia_pan"] < 0).sum())
print("Filas donde gasto_anemia_pan > gasto_total:", (df["gasto_anemia_pan"] > df["gasto_total"]).sum())

dataframes["siaf_gasto_devengado_distrital_2021_2025"] = df

Distritos con menos de 5 años reportados: 2
Años presentes: [np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025)]
Filas con gasto_total < 0: 0
Filas con gasto_anemia_pan < 0: 0
Filas donde gasto_anemia_pan > gasto_total: 0


# sien_reunis (panel de anemia, tu variable dependiente):

In [12]:
df = dataframes["sien_reunis_distrital_2020_2026"]
df["ubigeo"] = df["ubigeo"].astype(str).str.zfill(6)
df.rename(columns={"Año": "anio"}, inplace=True)

conteo_anios = df.groupby("ubigeo")["anio"].nunique()
print("Años presentes:", sorted(df["anio"].unique()))
print("Distritos con menos de 7 años reportados:", (conteo_anios < 7).sum())

inconsistentes = df["ninos_evaluados"] != (df["ninos_con_anemia"] + df["ninos_sin_anemia"])
print("Filas con evaluados != con_anemia + sin_anemia:", inconsistentes.sum())

prev_calc = df["ninos_con_anemia"] / df["ninos_evaluados"]
diff = (prev_calc - df["prevalencia_anemia"]).abs()
print("Máxima diferencia entre prevalencia reportada y calculada:", diff.max())

print("Departamentos únicos:", df["departamento"].str.strip().nunique(), "vs sin strip:", df["departamento"].nunique())

dataframes["sien_reunis_distrital_2020_2026"] = df

Años presentes: [np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025), np.int64(2026)]
Distritos con menos de 7 años reportados: 96
Filas con evaluados != con_anemia + sin_anemia: 0
Máxima diferencia entre prevalencia reportada y calculada: 1.1102230246251565e-16
Departamentos únicos: 25 vs sin strip: 25


# srtm (elevación y pendiente):

In [13]:
df = dataframes["srtm_distrital"]
df["ubigeo"] = df["ubigeo"].astype(str).str.zfill(6)

print("elevacion_media -> min:", df["elevacion_media"].min(), "max:", df["elevacion_media"].max())
print("pendiente_media -> min:", df["pendiente_media"].min(), "max:", df["pendiente_media"].max())
print("Nulos:", df[["elevacion_media", "pendiente_media"]].isna().sum().sum())

dataframes["srtm_distrital"] = df

elevacion_media -> min: 3.1187949664045345 max: 4815.83788180224
pendiente_media -> min: 0.9782390186503132 max: 35.36945723168035
Nulos: 0


# geobase (tabla maestra/universo de distritos, con geometría):

In [14]:
df = dataframes["geobase_distrital"]
df.rename(columns={"UBIGEO": "ubigeo"}, inplace=True)
df["ubigeo"] = df["ubigeo"].astype(str).str.zfill(6)

print("CRS:", df.crs)
print("Geometrías inválidas:", (~df.geometry.is_valid).sum())
print("Duplicados de ubigeo:", df["ubigeo"].duplicated().sum())

print("\nEjemplos de 'superficie':", df["superficie"].unique()[:5])
print("Ejemplos de 'pob_densidad_2020':", df["pob_densidad_2020"].unique()[:5])

dataframes["geobase_distrital"] = df

CRS: EPSG:4326
Geometrías inválidas: 1
Duplicados de ubigeo: 0

Ejemplos de 'superficie': <ArrowStringArray>
['153.78', '25.71', '357.09', '56.97', '143.43']
Length: 5, dtype: str
Ejemplos de 'pob_densidad_2020': <ArrowStringArray>
['201.43711796072299',   '13.9634383508363', '4.0465988966367004',
 '13.568544848165701', '6.8326012689116604']
Length: 5, dtype: str


# Limpiar superficie

In [15]:
df = dataframes["geobase_distrital"]

for col in ["superficie", "pob_densidad_2020"]:
    antes = df[col].copy()
    # quita espacios y reemplaza coma decimal por punto, por si vienen así
    limpio = df[col].astype(str).str.strip().str.replace(",", ".", regex=False)
    df[col] = pd.to_numeric(limpio, errors="coerce")
    fallidos = df[col].isna() & antes.notna()
    print(f"{col}: {fallidos.sum()} valores no se pudieron convertir a número")
    if fallidos.sum() > 0:
        print("Ejemplos que fallaron:", antes[fallidos].unique()[:5])

dataframes["geobase_distrital"] = df

superficie: 4 valores no se pudieron convertir a número
Ejemplos que fallaron: <ArrowStringArray>
['S.I.']
Length: 1, dtype: str
pob_densidad_2020: 4 valores no se pudieron convertir a número
Ejemplos que fallaron: <ArrowStringArray>
['S.I.']
Length: 1, dtype: str


# unir las tablas ESTÁTICAS (geobase como maestra, todas por ubigeo):

In [16]:
base_estatica = dataframes["geobase_distrital"].copy()

for nombre in ["esa_worldcover_distrital_2021", "open_buildings_distrital", "srtm_distrital"]:
    df = dataframes[nombre].drop(columns=["ANO_EJE", "anio"], errors="ignore")
    base_estatica = base_estatica.merge(df, on="ubigeo", how="left")

# jrc aparte: tiene menos filas, así que sus nulos los llenamos con 0 (= sin agua detectada)
df_jrc = dataframes["jrc_distrital"]
base_estatica = base_estatica.merge(df_jrc, on="ubigeo", how="left")
base_estatica[["pct_agua_permanente", "pct_agua_estacional"]] = \
    base_estatica[["pct_agua_permanente", "pct_agua_estacional"]].fillna(0)

print("Shape base estática:", base_estatica.shape)
print("Duplicados de ubigeo:", base_estatica["ubigeo"].duplicated().sum())
print("Nulos por columna:\n", base_estatica.isna().sum())

Shape base estática: (1889, 27)
Duplicados de ubigeo: 0
Nulos por columna:
 ubigeo                     0
departamento               0
provincia                  0
distrito                   0
region                     0
macroregion_inei           0
macroregion_minsa          0
capital                   15
latitude                  15
longitude                 15
altitude                  15
superficie                19
pob_densidad_2020         19
geometry                   0
pct_cultivo                0
pct_construido             0
pct_desnudo                0
pct_agua_visible           0
n_edificios                0
area_construida_m2         0
confianza_media            0
area_distrito_km2          0
densidad_edificios_km2     0
elevacion_media            0
pendiente_media            0
pct_agua_permanente        0
pct_agua_estacional        0
dtype: int64


# unir las tablas PANEL (distrito × año), con outer join para no perder ningún año:

In [17]:
base_panel = dataframes["sien_reunis_distrital_2020_2026"][["ubigeo", "anio", "ninos_evaluados",
    "ninos_con_anemia", "ninos_sin_anemia", "prevalencia_anemia"]].copy()

base_panel = base_panel.merge(
    dataframes["siaf_gasto_devengado_distrital_2021_2025"][["ubigeo", "anio", "gasto_total", "gasto_anemia_pan"]],
    on=["ubigeo", "anio"], how="outer"
)

base_panel = base_panel.merge(
    dataframes["renamu_distrital_2021_2024"][["ubigeo", "anio", "personal_total", "programa_anemia", "centro_salud_municipal"]],
    on=["ubigeo", "anio"], how="outer"
)

print("Shape base panel:", base_panel.shape)
print("Años en la base panel:", sorted(base_panel["anio"].unique()))
print("Duplicados ubigeo+anio:", base_panel.duplicated(subset=["ubigeo", "anio"]).sum())
print("Nulos por columna:\n", base_panel.isna().sum())

Shape base panel: (13148, 11)
Años en la base panel: [np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025), np.int64(2026)]
Duplicados ubigeo+anio: 0
Nulos por columna:
 ubigeo                       0
anio                         0
ninos_evaluados            119
ninos_con_anemia           119
ninos_sin_anemia           119
prevalencia_anemia         161
gasto_total               3696
gasto_anemia_pan          3696
personal_total            5812
programa_anemia           5622
centro_salud_municipal    5622
dtype: int64


# merge final: pega el contexto territorial estático a cada fila del panel:

In [18]:
cols_estaticas = [c for c in base_estatica.columns if c not in
                   ("geometry",)]  # dejamos la geometría fuera del dataset tabular final

df_final = base_panel.merge(base_estatica[cols_estaticas], on="ubigeo", how="left")

print("Shape final:", df_final.shape)
print("Distritos únicos:", df_final["ubigeo"].nunique(), "(esperado: 1889)")
print("Nulos por columna:\n", df_final.isna().sum())
display(df_final.head())

Shape final: (13148, 36)
Distritos únicos: 1891 (esperado: 1889)
Nulos por columna:
 ubigeo                       0
anio                         0
ninos_evaluados            119
ninos_con_anemia           119
ninos_sin_anemia           119
prevalencia_anemia         161
gasto_total               3696
gasto_anemia_pan          3696
personal_total            5812
programa_anemia           5622
centro_salud_municipal    5622
departamento                 9
provincia                    9
distrito                     9
region                       9
macroregion_inei             9
macroregion_minsa            9
capital                     97
latitude                    97
longitude                   97
altitude                    97
superficie                 125
pob_densidad_2020          125
pct_cultivo                  9
pct_construido               9
pct_desnudo                  9
pct_agua_visible             9
n_edificios                  9
area_construida_m2           9
confianza_media 

,ubigeo,anio,ninos_evaluados,ninos_con_anemia,ninos_sin_anemia,prevalencia_anemia,gasto_total,gasto_anemia_pan,personal_total,programa_anemia,...,pct_agua_visible,n_edificios,area_construida_m2,confianza_media,area_distrito_km2,densidad_edificios_km2,elevacion_media,pendiente_media,pct_agua_permanente,pct_agua_estacional
0,010101,2020,245.0,85.0,160.0,0.346939,NaN,NaN,NaN,NaN,...,0.000107,16298.0,1.306565e+06,0.783185,153.795602,105.971821,2423.265986,21.252837,0.019574,0.44859
1,010101,2021,234.0,71.0,163.0,0.303419,7.452997e+08,44747568.78,208.0,1.0,...,0.000107,16298.0,1.306565e+06,0.783185,153.795602,105.971821,2423.265986,21.252837,0.019574,0.44859
2,010101,2022,201.0,76.0,125.0,0.378109,7.182696e+08,10204248.58,198.0,0.0,...,0.000107,16298.0,1.306565e+06,0.783185,153.795602,105.971821,2423.265986,21.252837,0.019574,0.44859
3,010101,2023,837.0,160.0,677.0,0.191159,9.745361e+08,0.00,206.0,1.0,...,0.000107,16298.0,1.306565e+06,0.783185,153.795602,105.971821,2423.265986,21.252837,0.019574,0.44859
4,010101,2024,969.0,157.0,812.0,0.162023,1.002462e+09,0.00,268.0,1.0,...,0.000107,16298.0,1.306565e+06,0.783185,153.795602,105.971821,2423.265986,21.252837,0.019574,0.44859


# guardar

In [19]:
OUT_DIR = RAIZ / "data" / "clean" / "merged"
OUT_DIR.mkdir(parents=True, exist_ok=True)
out_path = OUT_DIR / "panel_distrital_final.csv"

df_final.to_csv(out_path, index=False)
print("Guardado en:", out_path)

Guardado en: c:\Users\JHOSSEP\Documents\REPO\causal-anemia-model\data\clean\merged\panel_distrital_final.csv
